# 🏎️ F1 2026 Full Season Predictor
**Predicts Top 10 + Podium for all 19 remaining 2026 races**

Run each cell top to bottom. Full training takes ~15 minutes.

In [ ]:
!pip install fastf1 xgboost shap plotly mlflow --quiet
print('Done!')

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import plotly.express as px
import joblib, os, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
import mlflow, mlflow.xgboost
import matplotlib.pyplot as plt

os.makedirs('/content/f1_cache',    exist_ok=True)
os.makedirs('/content/models',      exist_ok=True)
os.makedirs('/content/predictions', exist_ok=True)
fastf1.Cache.enable_cache('/content/f1_cache')
print(f'FastF1 {fastf1.__version__} | XGBoost {xgb.__version__}')
print('Imports ready!')

In [ ]:
DRIVERS_2026 = {
    'VER':'Red Bull Racing', 'HAD':'Red Bull Racing',
    'RUS':'Mercedes',        'ANT':'Mercedes',
    'LEC':'Ferrari',         'HAM':'Ferrari',
    'NOR':'McLaren',         'PIA':'McLaren',
    'ALO':'Aston Martin',    'STR':'Aston Martin',
    'GAS':'Alpine',          'DOO':'Alpine',
    'TSU':'Racing Bulls',    'LAW':'Racing Bulls',
    'HUL':'Haas',            'BEA':'Haas',
    'BOT':'Kick Sauber',     'BOR':'Kick Sauber',
    'COL':'Williams',        'LIN':'Williams',
}

REMAINING_2026 = [
    {'round':4,  'name':'Miami',         'circuit':'Miami',         'date':'2026-05-03'},
    {'round':5,  'name':'Canada',        'circuit':'Montreal',      'date':'2026-05-24'},
    {'round':6,  'name':'Monaco',        'circuit':'Monaco',        'date':'2026-06-07'},
    {'round':7,  'name':'Barcelona',     'circuit':'Catalunya',     'date':'2026-06-21'},
    {'round':8,  'name':'Austria',       'circuit':'Red Bull Ring', 'date':'2026-07-05'},
    {'round':9,  'name':'Britain',       'circuit':'Silverstone',   'date':'2026-07-19'},
    {'round':10, 'name':'Hungary',       'circuit':'Hungaroring',   'date':'2026-08-02'},
    {'round':11, 'name':'Belgium',       'circuit':'Spa',           'date':'2026-08-09'},
    {'round':12, 'name':'Netherlands',   'circuit':'Zandvoort',     'date':'2026-08-23'},
    {'round':13, 'name':'Italy',         'circuit':'Monza',         'date':'2026-09-06'},
    {'round':14, 'name':'Madrid',        'circuit':'Madrid',        'date':'2026-09-13'},
    {'round':15, 'name':'Azerbaijan',    'circuit':'Baku',          'date':'2026-09-26'},
    {'round':16, 'name':'Singapore',     'circuit':'Marina Bay',    'date':'2026-10-11'},
    {'round':17, 'name':'United States', 'circuit':'Austin',        'date':'2026-10-25'},
    {'round':18, 'name':'Mexico City',   'circuit':'Mexico City',   'date':'2026-11-01'},
    {'round':19, 'name':'Sao Paulo',     'circuit':'Interlagos',    'date':'2026-11-15'},
    {'round':20, 'name':'Las Vegas',     'circuit':'Las Vegas',     'date':'2026-11-22'},
    {'round':21, 'name':'Qatar',         'circuit':'Losail',        'date':'2026-11-29'},
    {'round':22, 'name':'Abu Dhabi',     'circuit':'Yas Marina',    'date':'2026-12-06'},
]

RESULTS_2026 = [
    {'Race':'Australia','Driver':'RUS','Position':1, 'GridPosition':1,  'Points':25},
    {'Race':'Australia','Driver':'ANT','Position':2, 'GridPosition':2,  'Points':18},
    {'Race':'Australia','Driver':'LEC','Position':3, 'GridPosition':5,  'Points':15},
    {'Race':'Australia','Driver':'HAM','Position':4, 'GridPosition':6,  'Points':12},
    {'Race':'Australia','Driver':'NOR','Position':5, 'GridPosition':4,  'Points':10},
    {'Race':'Australia','Driver':'VER','Position':6, 'GridPosition':20, 'Points':8},
    {'Race':'Australia','Driver':'BEA','Position':7, 'GridPosition':7,  'Points':6},
    {'Race':'Australia','Driver':'LIN','Position':8, 'GridPosition':8,  'Points':4},
    {'Race':'Australia','Driver':'BOR','Position':9, 'GridPosition':9,  'Points':2},
    {'Race':'Australia','Driver':'GAS','Position':10,'GridPosition':10, 'Points':1},
    {'Race':'China',    'Driver':'ANT','Position':1, 'GridPosition':2,  'Points':25},
    {'Race':'China',    'Driver':'RUS','Position':2, 'GridPosition':1,  'Points':18},
    {'Race':'China',    'Driver':'HAM','Position':3, 'GridPosition':4,  'Points':15},
    {'Race':'China',    'Driver':'LEC','Position':4, 'GridPosition':3,  'Points':12},
    {'Race':'China',    'Driver':'NOR','Position':5, 'GridPosition':5,  'Points':10},
    {'Race':'China',    'Driver':'VER','Position':6, 'GridPosition':8,  'Points':8},
    {'Race':'China',    'Driver':'PIA','Position':7, 'GridPosition':6,  'Points':6},
    {'Race':'China',    'Driver':'ALO','Position':8, 'GridPosition':7,  'Points':4},
    {'Race':'China',    'Driver':'GAS','Position':9, 'GridPosition':9,  'Points':2},
    {'Race':'China',    'Driver':'TSU','Position':10,'GridPosition':11, 'Points':1},
    {'Race':'Japan',    'Driver':'ANT','Position':1, 'GridPosition':1,  'Points':25},
    {'Race':'Japan',    'Driver':'PIA','Position':2, 'GridPosition':3,  'Points':18},
    {'Race':'Japan',    'Driver':'LEC','Position':3, 'GridPosition':4,  'Points':15},
    {'Race':'Japan',    'Driver':'RUS','Position':4, 'GridPosition':2,  'Points':12},
    {'Race':'Japan',    'Driver':'NOR','Position':5, 'GridPosition':5,  'Points':10},
    {'Race':'Japan',    'Driver':'HAM','Position':6, 'GridPosition':6,  'Points':8},
    {'Race':'Japan',    'Driver':'VER','Position':7, 'GridPosition':11, 'Points':6},
    {'Race':'Japan',    'Driver':'GAS','Position':8, 'GridPosition':7,  'Points':4},
    {'Race':'Japan',    'Driver':'HAD','Position':9, 'GridPosition':8,  'Points':2},
    {'Race':'Japan',    'Driver':'TSU','Position':10,'GridPosition':10, 'Points':1},
]

results_2026_df = pd.DataFrame(RESULTS_2026)
SEASON_WEIGHTS  = {2021:0.4, 2022:0.6, 2023:0.8, 2024:1.0, 2025:1.5, 2026:3.0}
print(f'Drivers: {len(DRIVERS_2026)} | Remaining races: {len(REMAINING_2026)}')
print('2026 known results:', len(results_2026_df), 'rows')

In [ ]:
SEASONS = [2021, 2022, 2023, 2024, 2025]
all_race_data = []

for year in SEASONS:
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        races = schedule[schedule['EventFormat'] != 'testing']
        print(f'\n{year}: {len(races)} races')
        for _, event in races.iterrows():
            rname = event['EventName']
            try:
                s = fastf1.get_session(year, rname, 'R')
                s.load(telemetry=False, weather=False, messages=False)
                res  = s.results.copy()
                laps = s.laps.copy()
                if res.empty: continue
                res['Position']     = pd.to_numeric(res['Position'],     errors='coerce')
                res['GridPosition'] = pd.to_numeric(res['GridPosition'], errors='coerce')
                res['Points']       = pd.to_numeric(res['Points'],       errors='coerce')
                laps['LapTimeSec']  = laps['LapTime'].dt.total_seconds()
                cl = laps[laps['LapTimeSec'].between(55,140) & laps['PitInTime'].isna() & laps['PitOutTime'].isna()].copy()
                ls = cl.groupby('Driver')['LapTimeSec'].agg(AvgLapTime='mean',BestLapTime='min',LapTimeStd='std',TotalLaps='count').reset_index()
                pc = laps[laps['PitInTime'].notna()].groupby('Driver').size().reset_index(name='PitStops')
                df_r = res[['Abbreviation','TeamName','Position','GridPosition','Points']].copy()
                df_r.columns = ['Driver','Team','Position','GridPosition','Points']
                df_r = df_r.merge(ls, on='Driver', how='left').merge(pc, on='Driver', how='left')
                df_r['PitStops']      = df_r['PitStops'].fillna(0).astype(int)
                df_r['PositionDelta'] = df_r['GridPosition'] - df_r['Position']
                df_r['Finished']      = (~df_r['Position'].isna()).astype(int)
                df_r['InPoints']      = (df_r['Position'] <= 10).astype(int)
                df_r['Podium']        = (df_r['Position'] <= 3).astype(int)
                df_r['Winner']        = (df_r['Position'] == 1).astype(int)
                df_r['Race']          = rname
                df_r['Season']        = year
                df_r['Weight']        = SEASON_WEIGHTS.get(year, 1.0)
                df_r['Circuit']       = event.get('Location', rname)
                all_race_data.append(df_r)
                print(f'  OK {rname}')
            except Exception as e:
                print(f'  SKIP {rname}: {str(e)[:50]}')
    except Exception as e:
        print(f'Error {year}: {e}')

historical_df = pd.concat(all_race_data, ignore_index=True)
print(f'\nTotal rows: {len(historical_df)}')
historical_df.to_csv('/content/historical_features.csv', index=False)
print('Saved!')

In [ ]:
df = historical_df.copy()
df = df.sort_values(['Driver','Season','Race']).reset_index(drop=True)
df['Last3AvgPos']  = df.groupby('Driver')['Position'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
df['Last3Points']  = df.groupby('Driver')['Points'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).sum())
df['CircuitAvgPos']= df.groupby(['Driver','Circuit'])['Position'].transform(lambda x: x.shift(1).expanding().mean())
df['DNF']          = (df['Finished']==0).astype(int)
df['DNFRate']      = df.groupby('Driver')['DNF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['DeltaTrend']   = df.groupby('Driver')['PositionDelta'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
team_pts = df.groupby(['Team','Race','Season'])['Points'].sum().reset_index()
team_pts['TeamLast3Pts'] = team_pts.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).sum())
df = df.merge(team_pts[['Team','Race','Season','TeamLast3Pts']], on=['Team','Race','Season'], how='left')
for col in ['Last3AvgPos','CircuitAvgPos']: df[col] = df[col].fillna(10)
for col in ['Last3Points','DeltaTrend','TeamLast3Pts']: df[col] = df[col].fillna(0)
df['DNFRate'] = df['DNFRate'].fillna(0.1)
print(f'Feature table: {df.shape}')
df.to_csv('/content/features_rolling.csv', index=False)
print('Rolling features saved!')

In [ ]:
r26 = results_2026_df.copy()
r26['Team']     = r26['Driver'].map(DRIVERS_2026)
r26['Season']   = 2026
r26['Weight']   = 3.0
r26['Podium']   = (r26['Position'] <= 3).astype(int)
r26['Winner']   = (r26['Position'] == 1).astype(int)
r26['InPoints'] = (r26['Position'] <= 10).astype(int)
r26['Finished'] = 1
r26['DNF']      = 0
r26['Circuit']  = r26['Race']
r26 = r26.sort_values(['Driver','Race'])
r26['Last3AvgPos']   = r26.groupby('Driver')['Position'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean()).fillna(10)
r26['Last3Points']   = r26.groupby('Driver')['Points'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).sum()).fillna(0)
r26['CircuitAvgPos'] = 10.0
r26['DNFRate']       = 0.05
r26['DeltaTrend']    = (r26['GridPosition'] - r26['Position']).fillna(0)
r26['PositionDelta'] = r26['DeltaTrend']
r26['TeamLast3Pts']  = r26.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).sum()).fillna(0)
r26['AvgLapTime']    = 90.0
r26['BestLapTime']   = 88.0
r26['LapTimeStd']    = 1.0
r26['TotalLaps']     = 50
r26['PitStops']      = 1
standings = r26.groupby('Driver')['Points'].sum().sort_values(ascending=False)
print('2026 standings after 3 races:')
print(standings.head(10))

In [ ]:
FEATURE_COLS = ['GridPosition','AvgLapTime','BestLapTime','LapTimeStd','PitStops',
                'Last3AvgPos','Last3Points','CircuitAvgPos','DNFRate','DeltaTrend',
                'TeamLast3Pts','TeamEncoded']

le_team   = LabelEncoder()
all_teams = pd.concat([df['Team'], r26['Team']]).dropna().unique()
le_team.fit(all_teams)

def encode_team(t):
    return le_team.transform([t])[0] if t in le_team.classes_ else 0

df['TeamEncoded']  = df['Team'].fillna('Unknown').map(encode_team)
r26['TeamEncoded'] = r26['Team'].fillna('Unknown').map(encode_team)

train_df = pd.concat([df, r26], ignore_index=True)
train_df = train_df.dropna(subset=FEATURE_COLS + ['Position'])
train_df['Position'] = pd.to_numeric(train_df['Position'], errors='coerce')
train_df = train_df.dropna(subset=['Position'])

X       = train_df[FEATURE_COLS].values
y_pos   = train_df['Position'].values.astype(int)
y_pod   = train_df['Podium'].values.astype(int)
y_top10 = train_df['InPoints'].values.astype(int)
W       = train_df['Weight'].values

print(f'Training set: {X.shape[0]} rows x {X.shape[1]} features')
print(f'Podium rate:  {y_pod.mean():.1%}')
print(f'Top10 rate:   {y_top10.mean():.1%}')

In [ ]:
neg_pod   = (y_pod==0).sum();   pos_pod   = (y_pod==1).sum()
neg_top10 = (y_top10==0).sum(); pos_top10 = (y_top10==1).sum()

mlflow.set_experiment('F1-2026-Predictor')
with mlflow.start_run(run_name='three_models_v1'):

    model_pos = xgb.XGBRegressor(n_estimators=300,max_depth=4,learning_rate=0.05,
                                  subsample=0.8,colsample_bytree=0.8,random_state=42)
    model_pos.fit(X, y_pos, sample_weight=W)
    mae = mean_absolute_error(y_pos, model_pos.predict(X))
    print(f'Model A Position Regressor  MAE: {mae:.2f} places')

    model_pod = xgb.XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.05,
                                   scale_pos_weight=round(neg_pod/pos_pod,2),
                                   subsample=0.8,colsample_bytree=0.8,
                                   random_state=42,eval_metric='logloss')
    sc_pod = cross_val_score(model_pod,X,y_pod,cv=5,scoring='accuracy',fit_params={'sample_weight':W})
    model_pod.fit(X, y_pod, sample_weight=W)
    print(f'Model B Podium Classifier   CV: {sc_pod.mean():.3f} +/- {sc_pod.std():.3f}')

    model_top10 = xgb.XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.05,
                                     scale_pos_weight=round(neg_top10/pos_top10,2),
                                     subsample=0.8,colsample_bytree=0.8,
                                     random_state=42,eval_metric='logloss')
    sc_t10 = cross_val_score(model_top10,X,y_top10,cv=5,scoring='accuracy',fit_params={'sample_weight':W})
    model_top10.fit(X, y_top10, sample_weight=W)
    print(f'Model C Top10 Classifier    CV: {sc_t10.mean():.3f} +/- {sc_t10.std():.3f}')

    mlflow.log_metrics({'position_mae':mae,'podium_cv':sc_pod.mean(),'top10_cv':sc_t10.mean()})
    mlflow.log_params({'n_estimators':300,'max_depth':4,'rows':len(X)})

joblib.dump(model_pos,    '/content/models/model_position.pkl')
joblib.dump(model_pod,    '/content/models/model_podium.pkl')
joblib.dump(model_top10,  '/content/models/model_top10.pkl')
joblib.dump(le_team,      '/content/models/team_encoder.pkl')
joblib.dump(FEATURE_COLS, '/content/models/feature_cols.pkl')
print('All 3 models saved!')

In [ ]:
explainer = shap.TreeExplainer(model_pod)
shap_vals = explainer.shap_values(X[:500])
shap.summary_plot(shap_vals,X[:500],feature_names=FEATURE_COLS,plot_type='bar',show=False)
plt.title('Feature importance - Podium model (2026)')
plt.tight_layout()
plt.savefig('/content/models/shap_importance.png',dpi=150)
plt.show()
print('SHAP saved!')

In [ ]:
CIRCUIT_GRIDS = {
    'Monaco':      ['LEC','ANT','RUS','VER','HAM','NOR','PIA','HAD','GAS','ALO','TSU','STR','BEA','HUL','LAW','BOT','BOR','COL','LIN','DOO'],
    'Monza':       ['ANT','RUS','NOR','PIA','LEC','HAM','VER','HAD','GAS','TSU','ALO','STR','BEA','HUL','LAW','BOT','BOR','COL','LIN','DOO'],
    'Marina Bay':  ['LEC','ANT','RUS','ALO','HAM','NOR','PIA','VER','HAD','GAS','TSU','STR','BEA','HUL','LAW','BOT','BOR','COL','LIN','DOO'],
    'Baku':        ['LEC','ANT','RUS','VER','HAM','NOR','PIA','HAD','GAS','ALO','TSU','STR','BEA','HUL','LAW','BOT','BOR','COL','LIN','DOO'],
    'Silverstone': ['NOR','PIA','ANT','RUS','LEC','HAM','VER','HAD','GAS','ALO','TSU','STR','BEA','HUL','LAW','BOT','BOR','COL','LIN','DOO'],
}
DEFAULT_GRID = ['ANT','RUS','LEC','HAM','NOR','PIA','VER','HAD','GAS','TSU','ALO','STR','BEA','HUL','LAW','BOT','BOR','COL','LIN','DOO']

def get_driver_features(driver, circuit):
    team   = DRIVERS_2026.get(driver,'Unknown')
    d26    = results_2026_df[results_2026_df['Driver']==driver]
    l3avg  = d26['Position'].tail(3).mean() if len(d26)>0 else 10.0
    l3pts  = d26['Points'].tail(3).sum()    if len(d26)>0 else 0.0
    hc     = df[(df['Driver']==driver) & (df['Circuit'].str.contains(circuit.split()[0],case=False,na=False))]
    cavg   = hc['Position'].mean() if len(hc)>0 else 10.0
    hd     = df[df['Driver']==driver].tail(10)
    return {
        'Driver':driver,'Team':team,
        'Last3AvgPos':round(l3avg,2),'Last3Points':round(l3pts,1),
        'CircuitAvgPos':round(cavg,2),
        'DNFRate':round(hd['DNFRate'].mean() if len(hd)>0 else 0.1, 3),
        'DeltaTrend':round(hd['DeltaTrend'].mean() if len(hd)>0 else 0.0, 2),
        'TeamLast3Pts':round(results_2026_df[results_2026_df['Driver'].map(DRIVERS_2026)==team]['Points'].sum(),1),
        'AvgLapTime':round(hd['AvgLapTime'].mean() if len(hd)>0 else 90.0, 3),
        'BestLapTime':round(hd['BestLapTime'].mean() if len(hd)>0 else 88.0, 3),
        'LapTimeStd':round(hd['LapTimeStd'].mean() if len(hd)>0 else 1.5, 3),
        'PitStops':1,'TeamEncoded':encode_team(team),
    }
print('Feature builder ready!')

In [ ]:
all_preds = []
for info in REMAINING_2026:
    grid = CIRCUIT_GRIDS.get(info['circuit'], DEFAULT_GRID)
    rows = []
    for gpos, driver in enumerate(grid, 1):
        f = get_driver_features(driver, info['circuit'])
        f['GridPosition'] = gpos
        f['Race']  = info['name']
        f['Round'] = info['round']
        f['Date']  = info['date']
        rows.append(f)
    rdf = pd.DataFrame(rows)
    X_r = rdf[FEATURE_COLS].values
    rdf['PredPos']    = np.clip(model_pos.predict(X_r),1,20).round(1)
    rdf['PodiumProb'] = model_pod.predict_proba(X_r)[:,1]
    rdf['Top10Prob']  = model_top10.predict_proba(X_r)[:,1]
    rdf = rdf.sort_values('PredPos').reset_index(drop=True)
    rdf['PredRank'] = range(1, len(rdf)+1)
    all_preds.append(rdf)

preds_df = pd.concat(all_preds, ignore_index=True)
preds_df.to_csv('/content/predictions/2026_season_predictions.csv', index=False)
print(f'Predictions done for {len(REMAINING_2026)} races!')

In [ ]:
print('=' * 62)
print('2026 F1 SEASON PREDICTIONS - ALL REMAINING RACES')
print('=' * 62)
for info in REMAINING_2026:
    rdf = preds_df[preds_df['Race']==info['name']].head(10)
    print(f"\nR{info['round']:02d} | {info['name']:<20} | {info['date']}")
    print(f"{'P':>2} {'DRV':>4} {'TEAM':<22} {'PODIUM%':>8} {'TOP10%':>7}")
    print('-' * 50)
    for _, row in rdf.iterrows():
        flag = ' *' if row['PodiumProb'] >= 0.5 else ''
        print(f"{int(row['PredRank']):>2} {row['Driver']:>4} {row['Team']:<22} {row['PodiumProb']:>7.1%} {row['Top10Prob']:>7.1%}{flag}")

In [ ]:
winners = preds_df[preds_df['PredRank']==1][['Race','Driver','Team','PodiumProb','Round']].sort_values('Round')
fig1 = px.bar(winners, x='Race', y='PodiumProb', color='Driver',
              title='2026 F1 - Predicted Race Winner per Grand Prix',
              text=[f"{d} {p:.0%}" for d,p in zip(winners['Driver'],winners['PodiumProb'])])
fig1.update_layout(xaxis_tickangle=-45, height=500)
fig1.write_html('/content/predictions/predicted_winners.html')
fig1.show()

top7  = ['ANT','RUS','LEC','HAM','NOR','PIA','VER']
rord  = [r['name'] for r in REMAINING_2026]
hmap  = preds_df[preds_df['Driver'].isin(top7)].pivot_table(index='Driver',columns='Race',values='PodiumProb',aggfunc='first').round(2)
hmap  = hmap[[c for c in rord if c in hmap.columns]]
fig2  = px.imshow(hmap, text_auto='.0%', color_continuous_scale='RdYlGn', zmin=0, zmax=1,
                  title='2026 Podium Probability Heatmap - Top 7 Drivers', aspect='auto')
fig2.update_layout(height=400)
fig2.write_html('/content/predictions/podium_heatmap.html')
fig2.show()

PM = {1:25,2:18,3:15,4:12,5:10,6:8,7:6,8:4,9:2,10:1}
crow = []
for drv, team in DRIVERS_2026.items():
    act  = results_2026_df[results_2026_df['Driver']==drv]['Points'].sum()
    pred = sum(PM.get(int(round(r)),0) for r in preds_df[preds_df['Driver']==drv]['PredRank'])
    crow.append({'Driver':drv,'Team':team,'Actual':act,'Predicted':pred,'Total':act+pred})
cdf = pd.DataFrame(crow).sort_values('Total',ascending=False)
cdf['Pos'] = range(1,len(cdf)+1)
fig3 = px.bar(cdf.head(10), x='Driver', y='Total', color='Team', text='Total',
              title='2026 Predicted Final Championship Standings')
fig3.write_html('/content/predictions/predicted_championship.html')
fig3.show()
print('\nPredicted Championship Top 10:')
print(cdf[['Pos','Driver','Team','Actual','Predicted','Total']].head(10).to_string(index=False))

In [ ]:
import shutil
from google.colab import files

pkg = '/content/download_package'
os.makedirs(pkg, exist_ok=True)

to_copy = [
    '/content/models/model_position.pkl',
    '/content/models/model_podium.pkl',
    '/content/models/model_top10.pkl',
    '/content/models/team_encoder.pkl',
    '/content/models/feature_cols.pkl',
    '/content/models/shap_importance.png',
    '/content/predictions/2026_season_predictions.csv',
    '/content/predictions/predicted_winners.html',
    '/content/predictions/podium_heatmap.html',
    '/content/predictions/predicted_championship.html',
]
for src in to_copy:
    if os.path.exists(src):
        shutil.copy(src, pkg)

shutil.make_archive('/content/f1_2026_outputs','zip', pkg)
files.download('/content/f1_2026_outputs.zip')
print('Download started!')

In [ ]:
def update_prediction(race_name, quali_order):
    info = next((r for r in REMAINING_2026 if r['name']==race_name), None)
    if not info: return print(f'{race_name} not found!')
    rows = []
    for gpos, driver in enumerate(quali_order, 1):
        f = get_driver_features(driver, info['circuit'])
        f['GridPosition'] = gpos
        rows.append(f)
    rdf = pd.DataFrame(rows)
    X_r = rdf[FEATURE_COLS].values
    rdf['PredPos']    = np.clip(model_pos.predict(X_r),1,20).round(1)
    rdf['PodiumProb'] = model_pod.predict_proba(X_r)[:,1]
    rdf['Top10Prob']  = model_top10.predict_proba(X_r)[:,1]
    rdf = rdf.sort_values('PredPos').reset_index(drop=True)
    rdf['PredRank'] = range(1,len(rdf)+1)
    print(f'\n{race_name} - Updated after qualifying')
    print(f"{'P':>2} {'DRV':>4} {'Grid':>5} {'Podium%':>8} {'Top10%':>7}")
    print('-'*35)
    for _, row in rdf.head(10).iterrows():
        flag = ' PODIUM' if row['PodiumProb']>=0.5 else ''
        print(f"{int(row['PredRank']):>2} {row['Driver']:>4} {int(row['GridPosition']):>5} {row['PodiumProb']:>7.1%} {row['Top10Prob']:>7.1%}{flag}")
    return rdf

print('update_prediction() ready!')
print('Example after Miami qualifying:')
print("update_prediction('Miami', ['ANT','RUS','LEC','NOR','PIA','HAM','VER','HAD','GAS','TSU','ALO','STR','BEA','HUL','LAW','BOT','BOR','COL','LIN','DOO'])")